# Vẽ lại đường hội tụ một giai đoạn & hai giai đoạn (`IBTrACS_6h`)

Notebook tách riêng từ notebook tổng để **đọc trực tiếp kết quả `.npz` đã lưu** trong `KetQua_3TapDuLieu/runs/` (không cần huấn luyện lại mô hình) và vẽ lại biểu đồ hội tụ của tập **`IBTrACS_6h`** với khung thang đo trục tung đồng bộ theo **`IBTrACS_CARQ`** (biên độ rộng tương đương và bước chia chuẩn `0.01`).

| Phần | Nội dung |

|---|---|

| 0 | Cấu hình đường dẫn `KetQua_3TapDuLieu/runs/`, danh sách seed, tầm dự báo và tuỳ chọn khung thang đo |

| 1 | Định nghĩa cấu hình `TWO` (một giai đoạn / hai giai đoạn) và hàm nạp lịch sử loss từ file `.npz` |

| 2 | Tự động đo biên độ trục tung của `IBTrACS_CARQ` và tính khung `(ymin, ymax)` tương ứng cho `IBTrACS_6h` |

| 3 | Vẽ riêng 3 biểu đồ `IBTrACS_6h` (`+6h`, `+12h`, `+24h`) với thang đo đã chuẩn hoá |

| 4 | Vẽ hình ghép 2 hàng (`IBTrACS_6h` và `IBTrACS_CARQ`) đồng bộ thang đo để đưa thẳng vào báo cáo |

## Thư viện

In [ ]:
import os

import re

import numpy as np

import pandas as pd

import matplotlib

import matplotlib.pyplot as plt

from matplotlib.ticker import MultipleLocator, FormatStrFormatter



try:

    from IPython.display import display

except ImportError:

    display = print



plt.rc('font', **{'size': 11})

pd.set_option('display.width', 200)

pd.set_option('display.max_columns', 20)



for name, mod in [('numpy', np), ('pandas', pd), ('matplotlib', matplotlib)]:

    print(f'{name:11s} {mod.__version__}')

## 0. Cấu hình chung

- `OUT_DIR`: thư mục kết quả của notebook tổng (`KetQua_3TapDuLieu`). Notebook tự tìm ở thư mục hiện tại hoặc thư mục cha.

- `TARGET_KEY`: tập cần vẽ lại thang đo (`'IBTrACS_6h'`).

- `REF_KEY`: tập tham chiếu biên độ trục tung (`'IBTrACS_CARQ'`).

- `SEED_CHOICE`: đặt `None` để tự động lấy seed đầu tiên đã chạy trong `SEEDS` (giống hệt mục 9.3 của notebook tổng), hoặc điền số cụ thể (vd. `20`, `42`).

- `MANUAL_YLIM`: giữ `None` để tự động khớp biên độ theo `IBTrACS_CARQ` và bước chia `0.01`, hoặc điền `(ymin, ymax)` thủ công nếu muốn tinh chỉnh thêm.

In [ ]:
FORECAST_LIST = [1, 2, 4]                   # số bước 6h: +6h, +12h, +24h

SEEDS = [20, 42, 123, 300, 2024]

SEED_CHOICE = None                          # None = lấy seed đầu tiên đã có file .npz trong SEEDS



TARGET_KEY = 'IBTrACS_6h'

REF_KEY = 'IBTrACS_CARQ'

Y_TICK_STEP = 0.01                          # bước chia vạch trục tung giống IBTrACS_CARQ



# Tuỳ chọn chỉnh tay (ymin, ymax) cho từng tầm nếu muốn ghi đè tính toán tự động

MANUAL_YLIM = {

    1: None,   # +6h,  vd: (0.105, 0.160)

    2: None,   # +12h, vd: (0.135, 0.185)

    4: None,   # +24h, vd: (0.195, 0.255)

}



def find_dir(d):

    for q in [d, os.path.join('..', d)]:

        if os.path.isdir(q):

            return q

    raise FileNotFoundError(f"Không tìm thấy thư mục '{d}' ở thư mục notebook hoặc thư mục cha")



OUT_DIR = find_dir('KetQua_3TapDuLieu')

RUN_DIR = os.path.join(OUT_DIR, 'runs')

print(f'OUT_DIR -> {os.path.abspath(OUT_DIR)}')

print(f'RUN_DIR -> {os.path.abspath(RUN_DIR)}')

## 1. Cấu hình mô hình & hàm đọc lịch sử chạy (`.npz`)

Giữ nguyên cấu trúc định danh `spec_id` và `run_path` của notebook tổng để trỏ đúng tới các file `.npz` đã lưu của bảng `TWO` (`Một giai đoạn` và `Hai giai đoạn`).

In [ ]:
DISP = ['dx', 'dy']

SPD = {'tự tính': ['spd_s', 'sin_s', 'cos_s'], 'có sẵn': ['spd_p', 'sin_p', 'cos_p']}

O_ARCH = {'layer': 'bilinear', 'ln': False, 'act': 'sigmoid', 'init': 'one'}

FULL_ARCH = {'layer': 'mneuron', 'ln': True, 'act': 'algebraic', 'init': 'scaled'}





def mnn(feats, opt='rmsprop', two_stage=False, **arch):

    return {'model': 'mnn', 'feats': list(feats), 'arch': {**O_ARCH, **arch}, 'opt': opt, 'two_stage': two_stage}





def full(src=None, two_stage=False):

    return mnn(DISP + (SPD[src] if src else []), 'adam', two_stage, **FULL_ARCH)





TWO_SPECS = [

    ('Một giai đoạn', 'tự tính', full('tự tính', two_stage=False)),

    ('Hai giai đoạn', 'tự tính', full('tự tính', two_stage=True)),

]





def spec_id(spec):

    s = [spec['model'], '-'.join(spec['feats']), spec['opt']]

    if spec['model'] == 'mnn':

        a = spec['arch']

        s += [a['layer'], f"ln{int(a['ln'])}", a['act'], a['init'], f"two{int(spec['two_stage'])}"]

    return re.sub(r'[^A-Za-z0-9_.-]', '', '_'.join(s))





def run_path(key, sid, fs, seed):

    return os.path.join(RUN_DIR, key, f'{sid}_fs{fs}_s{seed}.npz')





def load_two_stage_history(key, fs, seed_choice=SEED_CHOICE):

    """Nạp train_loss, val_loss, best_epoch của Một giai đoạn và Hai giai đoạn cho (key, fs)."""

    seeds_to_check = [seed_choice] if seed_choice is not None else SEEDS

    out = {}

    for lab, _, spec in TWO_SPECS:

        sid = spec_id(spec)

        found = None

        for s in seeds_to_check:

            p = run_path(key, sid, fs, s)

            if os.path.exists(p):

                found = (s, p)

                break

        if found is None:

            raise FileNotFoundError(f'Không tìm thấy file kết quả cho {key} | +{6*fs}h | {lab} (sid={sid})')

        seed_used, path = found

        z = np.load(path)

        out[lab] = {

            'seed': seed_used,

            'path': path,

            'train_loss': np.asarray(z['train_loss'], dtype=float),

            'val_loss': np.asarray(z['val_loss'], dtype=float),

            'best_epoch': int(z['best_epoch']),

        }

    return out





HIST = {k: {fs: load_two_stage_history(k, fs) for fs in FORECAST_LIST} for k in [TARGET_KEY, REF_KEY]}

for k in [TARGET_KEY, REF_KEY]:

    for fs in FORECAST_LIST:

        s1 = HIST[k][fs]['Một giai đoạn']

        s2 = HIST[k][fs]['Hai giai đoạn']

        print(f'{k:13s} | +{6*fs:2d}h | seed=({s1["seed"]},{s2["seed"]}) | '

              f'best_ep=({s1["best_epoch"]}, {s2["best_epoch"]})')

## 2. Tính khung thang đo chuẩn từ `IBTrACS_CARQ`

Đo biên độ trục tung tự nhiên của `IBTrACS_CARQ` ở từng tầm dự báo (`+6h`, `+12h`, `+24h`), sau đó đặt khung `(ymin, ymax)` cho `IBTrACS_6h` có **cùng độ rộng biên độ** (tối thiểu `0.050`–`0.060`) và bước chia vạch `0.01`.

In [ ]:
def get_data_bounds(hist_fs):

    vals = np.concatenate([

        hist_fs['Một giai đoạn']['train_loss'],

        hist_fs['Một giai đoạn']['val_loss'],

        hist_fs['Hai giai đoạn']['train_loss'],

        hist_fs['Hai giai đoạn']['val_loss'],

    ])

    return float(np.nanmin(vals)), float(np.nanmax(vals))





def compute_matched_ylims():

    ylims_target, ylims_ref, summary = {}, {}, []

    for fs in FORECAST_LIST:

        t_min, t_max = get_data_bounds(HIST[TARGET_KEY][fs])

        r_min, r_max = get_data_bounds(HIST[REF_KEY][fs])



        # Biên độ chuẩn của IBTrACS_CARQ (làm tròn lên bội số 0.005, tối thiểu 0.050)

        raw_ref_span = (r_max - r_min) * 1.08

        target_span = max(0.050, np.ceil(raw_ref_span / 0.005) * 0.005)



        # Khung cho IBTrACS_CARQ

        r_center = 0.5 * (r_min + r_max)

        ylims_ref[fs] = (round(r_center - 0.5 * target_span, 4), round(r_center + 0.5 * target_span, 4))



        # Khung cho IBTrACS_6h (ưu tiên MANUAL_YLIM nếu người dùng có đặt)

        if MANUAL_YLIM.get(fs) is not None:

            ymin, ymax = MANUAL_YLIM[fs]

        else:

            t_center = 0.5 * (t_min + t_max)

            span = max(target_span, (t_max - t_min) * 1.15)

            ymin = round(t_center - 0.5 * span, 4)

            ymax = round(t_center + 0.5 * span, 4)

        ylims_target[fs] = (ymin, ymax)



        summary.append({

            'Tầm': f'+{6*fs}h',

            'IBTrACS_6h gốc [min, max]': f'[{t_min:.4f}, {t_max:.4f}]',

            'Biên độ cũ': round(t_max - t_min, 4),

            'IBTrACS_6h khung mới': f'[{ymin:.4f}, {ymax:.4f}]',

            'Biên độ mới': round(ymax - ymin, 4),

            'IBTrACS_CARQ khung': f'[{ylims_ref[fs][0]:.4f}, {ylims_ref[fs][1]:.4f}]',

        })

    return ylims_target, ylims_ref, pd.DataFrame(summary)





YLIM_TARGET, YLIM_REF, scale_df = compute_matched_ylims()

display(scale_df)

## 3. Vẽ lại riêng tập `IBTrACS_6h` (`1 × 3`)

Xuất file `KetQua_3TapDuLieu/Hinh_hoi_tu_IBTrACS_6h_chuan_thang.png` với màu sắc, kiểu nét và bố cục đồng nhất với notebook tổng.

In [ ]:
def plot_converge_row(axes, key, ylim_map):

    colors = {'Một giai đoạn': '#4C78A8', 'Hai giai đoạn': '#E45756'}

    for c, fs in enumerate(FORECAST_LIST):

        ax = axes[c]

        for lab, _, _ in TWO_SPECS:

            d = HIST[key][fs][lab]

            col = colors[lab]

            ax.plot(d['train_loss'], color=col, lw=0.9, alpha=0.6)

            ax.plot(d['val_loss'], color=col, lw=2.0, label=f"{lab} (best ep {d['best_epoch']})")

        ax.set_ylim(*ylim_map[fs])

        ax.yaxis.set_major_locator(MultipleLocator(Y_TICK_STEP))

        ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))

        ax.set_title(f'{key} | +{6 * fs}h')

        ax.set_xlabel('epoch')

        ax.set_ylabel('MSE (chuẩn hoá)')

        ax.spines[['top', 'right']].set_visible(False)

        ax.grid(alpha=0.3)

        ax.legend(frameon=False, fontsize=9)





fig, axes = plt.subplots(1, len(FORECAST_LIST), figsize=(5 * len(FORECAST_LIST), 3.6))

plot_converge_row(axes, TARGET_KEY, YLIM_TARGET)

fig.tight_layout()

out_single = os.path.join(OUT_DIR, 'Hinh_hoi_tu_IBTrACS_6h_chuan_thang.png')

fig.savefig(out_single, dpi=300, bbox_inches='tight')

plt.show()

print(f'Đã lưu hình riêng IBTrACS_6h -> {os.path.abspath(out_single)}')

## 4. Vẽ hình ghép `IBTrACS_6h` & `IBTrACS_CARQ` đồng bộ thang đo (`2 × 3`)

Xuất thêm file `KetQua_3TapDuLieu/Hinh_hoi_tu_mot_hai_giai_doan_chuan_thang.png` đặt hai tập trên cùng tỷ lệ biên độ trục tung để tiện thay thế trực tiếp hình cũ trong báo cáo.

In [ ]:
fig, axes = plt.subplots(2, len(FORECAST_LIST), figsize=(5 * len(FORECAST_LIST), 6.8))

plot_converge_row(axes[0], TARGET_KEY, YLIM_TARGET)

plot_converge_row(axes[1], REF_KEY, YLIM_REF)

fig.tight_layout()

out_pair = os.path.join(OUT_DIR, 'Hinh_hoi_tu_mot_hai_giai_doan_chuan_thang.png')

fig.savefig(out_pair, dpi=300, bbox_inches='tight')

plt.show()

print(f'Đã lưu hình ghép 2 tập -> {os.path.abspath(out_pair)}')